### Build tournament training data


In [32]:
import pandas as pd
import numpy as np

### Load tournament results

In [33]:
tourney_men_df = pd.read_csv("../data/MNCAATourneyCompactResults.csv")
tourney_women_df = pd.read_csv("../data/WNCAATourneyCompactResults.csv")

### Filter seasons to match regular-season feature coverage

In [34]:
tourney_men_df = tourney_men_df[tourney_men_df["Season"] >= 2003].reset_index(drop=True)
tourney_women_df = tourney_women_df[tourney_women_df["Season"] >= 2010].reset_index(drop=True)

### Build historical rows

In [35]:
def build_historical_matchups(tourney_df):
    forward_df = tourney_df.copy()
    reverse_df = tourney_df.copy()

    forward_df["Team1ID"] = forward_df["WTeamID"]
    forward_df["Team2ID"] = forward_df["LTeamID"]
    forward_df["Target"] = 1

    reverse_df["Team1ID"] = reverse_df["LTeamID"]
    reverse_df["Team2ID"] = reverse_df["WTeamID"]
    reverse_df["Target"] = 0

    hist_df = pd.concat([forward_df, reverse_df], ignore_index=True)

    return hist_df[["Season", "DayNum", "WLoc", "NumOT", "Team1ID", "Team2ID", "Target"]].copy()

In [36]:
men_hist = build_historical_matchups(tourney_men_df)
women_hist = build_historical_matchups(tourney_women_df)

In [37]:
men_hist.head()

,Season,DayNum,WLoc,NumOT,Team1ID,Team2ID,Target
0,2003,134,N,1,1421,1411,1
1,2003,136,N,0,1112,1436,1
2,2003,136,N,0,1113,1272,1
3,2003,136,N,0,1141,1166,1
4,2003,136,N,1,1143,1301,1


In [38]:
women_hist.head()

,Season,DayNum,WLoc,NumOT,Team1ID,Team2ID,Target
0,2010,138,N,0,3124,3201,1
1,2010,138,N,0,3173,3395,1
2,2010,138,H,0,3181,3214,1
3,2010,138,H,0,3199,3256,1
4,2010,138,N,0,3207,3265,1


### Load season features

In [39]:
team_features_men = pd.read_csv("../data/processed/m_team_season_features.csv")
team_features_women = pd.read_csv("../data/processed/w_team_season_features.csv")

### Merge team season features into matchup rows

In [40]:
def merge_team_features(matchups_df, team_features):
    df = matchups_df.copy()

    team1_features = team_features.add_prefix("Team1_")
    team2_features = team_features.add_prefix("Team2_")

    df = df.merge(
        team1_features,
        left_on=["Season", "Team1ID"],
        right_on=["Team1_Season", "Team1_TeamID"],
        how="left"
    )

    df = df.merge(
        team2_features,
        left_on=["Season", "Team2ID"],
        right_on=["Team2_Season", "Team2_TeamID"],
        how="left"
    )

    return df

In [41]:
men_matchups = merge_team_features(men_hist, team_features_men)
women_matchups = merge_team_features(women_hist, team_features_women)

### Engineer matchup features

### Feature engineering
- WinPctDiff
- WinPctGap
- SeedNumDiff
- SeedGap
- NetRatingDiff
- NetRatingGap

In [42]:
def add_matchup_features(df, rating_col=None):
    df = df.copy()

    # Win %
    df["WinPctDiff"] = df["Team1_WinPct"] - df["Team2_WinPct"]
    df["WinPctGap"] = np.abs(df["WinPctDiff"])

    # Seed
    df["SeedNumDiff"] = df["Team1_SeedNum"] - df["Team2_SeedNum"]
    df["SeedGap"] = np.abs(df["SeedNumDiff"])

    # Net rating
    df["NetRatingDiff"] = df["Team1_AvgNetRating"] - df["Team2_AvgNetRating"]
    df["NetRatingGap"] = np.abs(df["NetRatingDiff"])

    # Efficiency
    df["OffEffDiff"] = df["Team1_AvgOffEfficiency"] - df["Team2_AvgOffEfficiency"]
    df["DefEffDiff"] = df["Team1_AvgDefEfficiency"] - df["Team2_AvgDefEfficiency"]

    # Margin
    df["MarginDiff"] = df["Team1_AvgMarginScore"] - df["Team2_AvgMarginScore"]

    # Rebounding
    df["ReboundPctDiff"] = df["Team1_AvgReboundPct"] - df["Team2_AvgReboundPct"]
    df["ReboundMarginDiff"] = df["Team1_AvgReboundMargin"] - df["Team2_AvgReboundMargin"]

    # Turnovers
    df["TurnoverPctDiff"] = df["Team1_AvgTurnoverPct"] - df["Team2_AvgTurnoverPct"]
    df["TurnoverMarginDiff"] = df["Team1_AvgTurnoverMargin"] - df["Team2_AvgTurnoverMargin"]
    df["AssistTurnoverRatioDiff"] = (
            df["Team1_AvgAssistTurnoverRto"] - df["Team2_AvgAssistTurnoverRto"]
    )

    # Shooting
    df["FGPctDiff"] = df["Team1_AvgFieldGoalsPct"] - df["Team2_AvgFieldGoalsPct"]
    df["ThreePctDiff"] = df["Team1_AvgThreePointsPct"] - df["Team2_AvgThreePointsPct"]
    df["FTPctDiff"] = df["Team1_AvgFreeThrowPct"] - df["Team2_AvgFreeThrowPct"]

    # Pace
    df["PossessionsDiff"] = df["Team1_AvgPossessions"] - df["Team2_AvgPossessions"]

    # Ranking
    if rating_col is not None:
        df["RankingDiff"] = df[f"Team1_{rating_col}"] - df[f"Team2_{rating_col}"]
        df["RankingDiff"] = df["RankingDiff"].fillna(0)

    return df

In [44]:
men_matchups = add_matchup_features(men_matchups, rating_col="MasseyOrdinalRank")
women_matchups = add_matchup_features(women_matchups, rating_col="WMasseyRating")

### Define model feature columns

### Build dataset for training

For training we need:
 - Team identifiers
  - seasons
  - target variable
  - engineered features

Right now we only have 12 features, but we can play around and see how we can add or remove them as we go. I believe we have a total of 100(?) features to use =)

In [45]:
TRAINING_FEATURE_COLUMNS = [
    "Season",
    "Team1ID",
    "Team2ID",
    "Target",
    "WinPctDiff",
    "SeedNumDiff",
    "NetRatingDiff",
    "OffEffDiff",
    "DefEffDiff",
    "MarginDiff",
    "ReboundPctDiff",
    "TurnoverPctDiff",
    "FGPctDiff",
    "ThreePctDiff",
    "FTPctDiff",
    "RankingDiff",
    "PossessionsDiff",
    "TurnoverMarginDiff",
    "ReboundMarginDiff",
    "AssistTurnoverRatioDiff",
]

### Build training datasets

In [46]:
men_training_df = men_matchups[TRAINING_FEATURE_COLUMNS].copy()
women_training_df = women_matchups[TRAINING_FEATURE_COLUMNS].copy()

In [47]:
men_training_df["RankingDiff"] = men_training_df["RankingDiff"].fillna(0)
men_training_df["AssistTurnoverRatioDiff"] = men_training_df["AssistTurnoverRatioDiff"].fillna(0)

In [48]:
women_training_df["RankingDiff"] = women_training_df["RankingDiff"].fillna(0)
women_training_df["AssistTurnoverRatioDiff"] = women_training_df["AssistTurnoverRatioDiff"].fillna(0)

In [49]:
print("Men training dataset shape:", men_training_df.shape)
print("Men missing values:", men_training_df.isna().sum().sum())
print("Men target distribution:")
print(men_training_df["Target"].value_counts(normalize=True))

Men training dataset shape: (2898, 20)
Men missing values: 0
Men target distribution:
Target
1    0.5
0    0.5
Name: proportion, dtype: float64


In [50]:
print("Women training dataset shape:", women_training_df.shape)
print("Women missing values:", women_training_df.isna().sum().sum())
print("Women target distribution:")
print(women_training_df["Target"].value_counts(normalize=True))

Women training dataset shape: (1922, 20)
Women missing values: 0
Women target distribution:
Target
1    0.5
0    0.5
Name: proportion, dtype: float64


In [51]:
print("Men missing by column:")
print(men_training_df.isna().sum().sort_values(ascending=False))

Men missing by column:
Season                     0
Team1ID                    0
Team2ID                    0
Target                     0
WinPctDiff                 0
SeedNumDiff                0
NetRatingDiff              0
OffEffDiff                 0
DefEffDiff                 0
MarginDiff                 0
ReboundPctDiff             0
TurnoverPctDiff            0
FGPctDiff                  0
ThreePctDiff               0
FTPctDiff                  0
RankingDiff                0
PossessionsDiff            0
TurnoverMarginDiff         0
ReboundMarginDiff          0
AssistTurnoverRatioDiff    0
dtype: int64


In [52]:
print("Women missing by column:")
print(women_training_df.isna().sum().sort_values(ascending=False))

Women missing by column:
Season                     0
Team1ID                    0
Team2ID                    0
Target                     0
WinPctDiff                 0
SeedNumDiff                0
NetRatingDiff              0
OffEffDiff                 0
DefEffDiff                 0
MarginDiff                 0
ReboundPctDiff             0
TurnoverPctDiff            0
FGPctDiff                  0
ThreePctDiff               0
FTPctDiff                  0
RankingDiff                0
PossessionsDiff            0
TurnoverMarginDiff         0
ReboundMarginDiff          0
AssistTurnoverRatioDiff    0
dtype: int64


### Save outputs

In [53]:
men_training_df.to_csv("../data/processed/m_tournament_training_dataset.csv", index=False)
women_training_df.to_csv("../data/processed/w_tournament_training_dataset.csv", index=False)